# Predict on Test Data and Create Submission CSV

This notebook loads trained model weights, makes predictions on test data, and saves the results in submission CSV format.


In [15]:
# Environment detection and path setup
import sys
from pathlib import Path

# Detect if running in Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab-specific overrides
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/AML-ETH-Project2')
    src_path = BASE_PATH / 'src'
else:
    current_dir = Path.cwd()
    BASE_PATH = current_dir.parent
    src_path = BASE_PATH / 'src'

print(f"Running in {'Colab' if IN_COLAB else 'local'} environment")
print(f"Base path: {BASE_PATH}")
print(f"src path: {src_path}")

# Add src to sys.path
sys.path.append(str(src_path))
print(f"Path '{src_path}' added to sys.path.")

# Define paths
weights_dir = BASE_PATH / 'weights'
weights_path = weights_dir / 'unet_weights.pth'
test_data_path = BASE_PATH / 'data' / 'raw' / 'test.pkl'
submissions_dir = BASE_PATH / 'submissions'
submissions_dir.mkdir(parents=True, exist_ok=True)


Running in local environment
Base path: /users/lmantel/lab
src path: /users/lmantel/lab/src
Path '/users/lmantel/lab/src' added to sys.path.


In [16]:
# Imports
import pickle
import gzip
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.notebook import tqdm

# Project-specific imports
from models.UNet import UNet, initialize_weights
from utils.utilities import load_zipped_pickle, get_mask_from_image

print('Imports OK')


Imports OK


### Helper Functions

Use the exact functions from task3.ipynb


In [17]:
def preprocess_test_data(data):
    video_frames = []
    names = []
    for item in tqdm(data):
        video = item['video']
        video = video.astype(np.float32).transpose((2, 0, 1))
        video = np.expand_dims(video, axis=3)
        video_frames += list(video)
        names += [item['name'] for _ in video]
    return names, video_frames

def get_sequences(arr):
    first_indices, last_indices, lengths = [], [], []
    n, i = len(arr), 0
    arr = [0] + list(arr) + [0]
    for index, value in enumerate(arr[:-1]):
        if arr[index+1]-arr[index] == 1:
            first_indices.append(index)
        if arr[index+1]-arr[index] == -1:
            last_indices.append(index)
    lengths = list(np.array(last_indices)-np.array(first_indices))
    return first_indices, lengths


### Load Model and Weights


In [18]:
# Initialize model
model = UNet(n_channels=1, n_classes=2)
initialize_weights(model)

# Load weights
if not weights_path.exists():
    raise FileNotFoundError(f'Weights not found at: {weights_path}')

model.load_state_dict(torch.load(str(weights_path), map_location='cpu'))
print(f'Weights loaded from: {weights_path}')

# Set device and move model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
model = model.to(device)
model.eval()


Weights loaded from: /users/lmantel/lab/weights/unet_weights.pth
Using device: cuda


/tmp/ipykernel_71832/1164730016.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(str(weights_path), map_location='cpu'))


UNet(
  (inc): DoubleConv(
    (double_conv): Sequential(
      (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (down1): Down(
    (pool_conv): Sequential(
      (0): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (1): DoubleConv(
        (double_conv): Sequential(
          (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    

### Load Test Data


In [19]:
# Load test data
if not test_data_path.exists():
    raise FileNotFoundError(f'Test data not found at: {test_data_path}')

test_data = load_zipped_pickle(str(test_data_path))
print(f'Loaded {len(test_data)} test samples from {test_data_path}')


Loaded 20 test samples from /users/lmantel/lab/data/raw/test.pkl


### Preprocess Test Data

Use preprocess_test_data to get individual frames


In [20]:
# Preprocess test data using the function from task3.ipynb
test_names, test_videos = preprocess_test_data(test_data)
print(f'Preprocessed {len(test_names)} frames from {len(test_data)} videos')


  0%|          | 0/20 [00:00<?, ?it/s]

Preprocessed 1507 frames from 20 videos


### Make Predictions and Create 3D Masks

For each video, predict masks for all frames and create a 3D mask, then flatten it.


In [ ]:
# Model input size (as used during training)
H, W = 256, 256

# Group frames by video name
video_dict = {}
for name, frame in zip(test_names, test_videos):
    if name not in video_dict:
        video_dict[name] = []
    video_dict[name].append(frame)

# Process each video to create 3D masks
ids = []
values = []

for name in tqdm(sorted(video_dict.keys()), desc="Processing videos"):
    frames = video_dict[name]
    
    # Get original video dimensions from test_data
    video_item = next(item for item in test_data if item['name'] == name)
    H_orig, W_orig, T = video_item['video'].shape
    
    # Create 3D mask: (H, W, T)
    mask_3d = np.zeros((H_orig, W_orig, T), dtype=bool)
    
    # Predict mask for each frame
    for t, frame_data in enumerate(frames):
        # frame_data has shape (H_orig, W_orig, 1) from preprocess_test_data
        frame = frame_data[:, :, 0].astype(np.float32) / 255.0  # Normalize to [0, 1]
        
        # Get prediction mask using get_mask_from_image
        mask_pred = get_mask_from_image(frame, model, H, W)  # Returns (H_orig, W_orig) binary mask
        
        # Store in 3D mask (convert to boolean)
        mask_3d[:, :, t] = mask_pred > 0
    
    # Flatten the 3D mask
    flattened_mask = mask_3d.flatten().astype(int)
    
    # Get sequences from flattened mask
    first_indices, lengths = get_sequences(flattened_mask)
    
    # Create one row per sequence with id = name_i and value = "[flattenedIdx, len]"
    for i, (idx, length) in enumerate(zip(first_indices, lengths)):
        ids.append(f"{name}_{i}")
        values.append([int(idx), int(length)])

print(f'Created {len(ids)} rows for {len(video_dict)} videos')


Processing videos:   0%|          | 0/20 [00:00<?, ?it/s]

Created 931613 rows for 20 videos


### Create and Save Submission CSV

Use the exact format from task3.ipynb


In [22]:
# Create DataFrame in submission format (exact format from task3.ipynb)
df = pd.DataFrame({"id": ids, "value": [list(map(int, minili)) for minili in values]})
df.to_csv(str(submissions_dir / 'submission.csv'), index=False)
print(f'Submission CSV saved to: {submissions_dir / "submission.csv"}')
print(f'\nSubmission preview:')
print(df.head(10))


Submission CSV saved to: /users/lmantel/lab/submissions/submission.csv

Submission preview:
             id          value
0  0MVRNDWR1G_0  [13192975, 1]
1  0MVRNDWR1G_1  [13193038, 1]
2  0MVRNDWR1G_2  [13193101, 1]
3  0MVRNDWR1G_3  [13235185, 1]
4  0MVRNDWR1G_4  [13235248, 1]
5  0MVRNDWR1G_5  [13235311, 1]
6  0MVRNDWR1G_6  [13277395, 1]
7  0MVRNDWR1G_7  [13277458, 1]
8  0MVRNDWR1G_8  [13277521, 1]
9  0MVRNDWR1G_9  [13318282, 1]
